<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
nn.Module로 모델 구축하기
</div>

# accelerator 감지하고 device 결정

**목표**: CUDA/MPS/CPU 중 사용 가능한 가속기를 한 줄로 자동 감지해 `device` 변수에 담는다.

In [ ]:
import torch
from torch import nn

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
device

# `NeuralNetwork` 클래스 정의

**목표**: `nn.Module`을 상속한 FashionMNIST 분류기 클래스를 작성한다. `__init__`에 레이어를 등록하고 `forward`에 흐름을 적는다.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()                       # 반드시 첫 줄
        self.flatten = nn.Flatten()              # [N, 28, 28] -> [N, 784]
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512), nn.ReLU(),
            nn.Linear(512, 512),     nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


# 인스턴스 생성 + device 이동 + 트리 출력

**목표**: 모델 인스턴스를 만들어 `device`에 올리고, `print(model)`로 자동 생성된 트리 표현을 확인한다.

In [ ]:
model = NeuralNetwork().to(device)
print(model)

# 더미 입력 한 장으로 추론

**목표**: 무작위 입력 한 장으로 `model(X)`를 호출해 logits를 얻고, Softmax + argmax로 예측 클래스를 추출한다.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
print(f"입력 X shape: {tuple(X.shape)}  | dtype: {X.dtype}  | device: {X.device}")
print(f"X 값 범위: min={X.min().item():.3f}  max={X.max().item():.3f}")

In [ ]:
logits = model(X)                                # ← 반드시 model(X), forward 직호출 금지
pred_probab = nn.Softmax(dim=1)(logits)          # [1, 10], 행 합 = 1
y_pred = pred_probab.argmax(1)                   # [1], 가장 큰 클래스 인덱스
print(f"logits shape: {tuple(logits.shape)}")
print(f"확률 행 합: {pred_probab.sum(dim=1).item():.4f}  (1에 가까워야 함)")
print(f"Predicted class: {y_pred.item()}  (0~9 중 하나)")

# 5개 레이어 분해해 shape 추적

**목표**: 3장짜리 미니배치를 만들어 `Flatten → Linear → ReLU`를 차례로 적용하며 shape가 어떻게 변하는지 print로 확인한다.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
input_image = torch.rand(3, 28, 28)
print(f"input_image: {tuple(input_image.size())}")

In [ ]:
# 데이터 시각화 — 입력 3장이 무엇처럼 생겼는지 먼저 본다
fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for i, ax in enumerate(axes):
    ax.imshow(input_image[i], cmap="gray")
    ax.set_title(f"sample {i}")
    ax.axis("off")
plt.suptitle("Input mini batch (3 piece of random noise)", y=1.05)
plt.show()

In [ ]:
# (1) Flatten
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(f"after Flatten     : {tuple(flat_image.size())}   ← 배치 축 3 유지, 28x28=784")

In [ ]:
# (2) Linear
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)
print(f"after Linear(784,20): {tuple(hidden1.size())}    ← (3, 20)")

In [ ]:
# (3) ReLU
print(f"Before ReLU min/max: {hidden1.min().item():.3f} / {hidden1.max().item():.3f}")
hidden1 = nn.ReLU()(hidden1)
print(f"After  ReLU min/max: {hidden1.min().item():.3f} / {hidden1.max().item():.3f}   ← min은 0 이상")

### 단계 6 — `nn.Sequential`로 동등 표현

**목표**: 같은 레이어들을 `Sequential` 컨테이너 한 줄로 묶어 호출하면 단계 5의 4단계가 한 번에 처리됨을 확인한다.

**코드**:


In [ ]:
seq_modules = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 20),
    nn.ReLU(),
    nn.Linear(20, 10),
)

input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(f"seq_modules output shape: {tuple(logits.size())}   ← (3, 10)")

# `named_parameters()` + 총 파라미터 수

**목표**: `model.named_parameters()`로 6줄의 (이름, shape)을 출력하고, `sum(p.numel())`으로 학습 가능 파라미터 총합 669,706을 직접 계산해 §4-C 도식의 산식과 대조한다.

In [ ]:
print("이름                                      | shape")
print("-" * 60)
for name, param in model.named_parameters():
    print(f"{name:40s} | {tuple(param.shape)}")

In [ ]:
total = sum(p.numel() for p in model.parameters())
print("-" * 60)
print(f"총 학습 가능 파라미터: {total:,}")

In [ ]:
# 수동 검산 — §4-C 산식과 비교
manual = 512*784 + 512 + 512*512 + 512 + 10*512 + 10
print(f"수동 계산  : 512*784 + 512 + 512*512 + 512 + 10*512 + 10 = {manual:,}")
assert total == manual, "두 값이 같아야 합니다!"
print("✓ 코드와 수동 계산이 일치")